In [1]:
import pandas as pd

# 1. Cargar las tablas (usamos '../' porque los datos están un nivel más arriba)
repo = pd.read_parquet('../repository.parquet')
history = pd.read_parquet('../source_markdown_file_history.parquet')
version = pd.read_parquet('../source_markdown_file_version.parquet')
snapshot = pd.read_parquet('../source_markdown_file_snapshot.parquet')

# 2. Relacionar las tablas usando las claves del diccionario
# Unimos version con history
df = pd.merge(version, history, on='source_markdown_file_history_id', how='left')

# Unimos el resultado con snapshot para tener el contenido y el repository_id
df = pd.merge(df, snapshot, on='source_markdown_file_snapshot_id', how='left')

# Unimos el resultado con repository para tener los datos del repo
df = pd.merge(df, repo, on='repository_id', how='left')

# 3. Informar cantidades únicas
cantidad_repos = df['repository_id'].nunique()
cantidad_historias = df['source_markdown_file_history_id'].nunique()
cantidad_versiones = df['source_markdown_file_version_id'].nunique()

print(f"Cantidad de repositorios: {cantidad_repos}")
print(f"Cantidad de historias: {cantidad_historias}")
print(f"Cantidad de versiones: {cantidad_versiones}")

# 4. Identificar historias con una sola versión vs múltiples
historias_una_version = (history['version_count'] == 1).sum()
historias_multiples = (history['version_count'] >= 2).sum()

print(f"\nHistorias con una sola versión: {historias_una_version}")
print(f"Historias con al menos dos versiones: {historias_multiples}")

# Conservar solo los archivos con al menos dos versiones para analizar su evolución
df_analisis = df[df['version_count'] >= 2].copy()

Cantidad de repositorios: 262
Cantidad de historias: 604
Cantidad de versiones: 2820

Historias con una sola versión: 198
Historias con al menos dos versiones: 406


In [4]:
# 1. Convertir la columna de fechas a un formato que Python entienda (datetime)
# (Usamos utc=True por si hay diferentes zonas horarias)
df_analisis['committed_at'] = pd.to_datetime(df_analisis['committed_at'], utc=True)

# 2. Ordenar los datos: primero agrupamos por archivo, y luego ordenamos por la fecha de la versión
df_analisis = df_analisis.sort_values(by=['source_markdown_file_history_id', 'committed_at'])

# 3. Calcular el tiempo transcurrido entre una versión y la versión inmediatamente anterior
df_analisis['tiempo_desde_anterior'] = df_analisis.groupby('source_markdown_file_history_id')['committed_at'].diff()

# 4. Convertir ese tiempo a "días" para que sea más fácil de analizar y graficar después
df_analisis['dias_desde_anterior'] = df_analisis['tiempo_desde_anterior'].dt.total_seconds() / (24 * 3600)

# 5. Mostrar una muestra de cómo quedó estructurado el tiempo de evolución
print("Muestra de la evolución temporal (primeros 10 registros):")
display(df_analisis[['source_markdown_file_history_id', 'rank', 'committed_at', 'dias_desde_anterior']].head(10))

# 6. Guardar este dataset procesado para usarlo en el siguiente Notebook de gráficos
df_analisis.to_parquet('dataset_evolucion_preparado.parquet')
print("\n¡Dataset listo y guardado como 'dataset_evolucion_preparado.parquet'!")

Muestra de la evolución temporal (primeros 10 registros):


,source_markdown_file_history_id,rank,committed_at,dias_desde_anterior
1,2,1,2026-02-17 15:11:01+00:00,NaN
2,2,2,2026-02-18 08:35:17+00:00,0.725185
3,2,3,2026-05-29 09:15:49+00:00,100.028148
4,2,4,2026-06-17 18:15:03+00:00,19.374468
5,3,1,2026-02-09 22:57:17+00:00,NaN
6,3,2,2026-02-09 23:09:51+00:00,0.008727
7,3,3,2026-02-09 23:27:44+00:00,0.012419
8,4,1,2026-06-04 08:47:29+00:00,NaN
9,4,2,2026-06-04 11:15:04+00:00,0.102488
10,5,1,2026-05-29 14:47:20+00:00,NaN



¡Dataset listo y guardado como 'dataset_evolucion_preparado.parquet'!


In [5]:
import yaml
import numpy as np

# 1. Definir funciones seguras para contar
def contar_palabras(texto):
    if pd.isna(texto) or not isinstance(texto, str) or not texto.strip():
        return np.nan
    return len(texto.split())

def contar_claves(texto):
    if pd.isna(texto) or not isinstance(texto, str) or not texto.strip():
        return np.nan
    try:
        parsed = yaml.safe_load(texto)
        if isinstance(parsed, dict):
            return len(parsed.keys())
        return 0
    except:
        return np.nan # Manejo de errores de interpretación del YAML

# 2. Calcular medidas base en la versión actual
df_analisis['longitud_body'] = df_analisis['body'].apply(contar_palabras)
df_analisis['tamano_frontmatter'] = df_analisis['frontmatter'].apply(contar_claves)

# 3. Obtener los valores de la versión anterior (el dataset ya está ordenado por tu celda anterior)
df_analisis['longitud_body_anterior'] = df_analisis.groupby('source_markdown_file_history_id')['longitud_body'].shift(1)
df_analisis['tamano_frontmatter_anterior'] = df_analisis.groupby('source_markdown_file_history_id')['tamano_frontmatter'].shift(1)

# 4. Calcular los cambios (Transiciones)
df_analisis['cambio_longitud_body'] = df_analisis['longitud_body'] - df_analisis['longitud_body_anterior']
df_analisis['magnitud_cambio_body'] = df_analisis['cambio_longitud_body'].abs()
df_analisis['cambio_tamano_frontmatter'] = df_analisis['tamano_frontmatter'] - df_analisis['tamano_frontmatter_anterior']

# Mostrar una muestra para verificar los cálculos
display(df_analisis[['source_markdown_file_history_id', 'rank', 'longitud_body', 'cambio_longitud_body', 'tamano_frontmatter']].head(10))

,source_markdown_file_history_id,rank,longitud_body,cambio_longitud_body,tamano_frontmatter
1,2,1,105,NaN,7.0
2,2,2,105,0.0,7.0
3,2,3,104,-1.0,7.0
4,2,4,104,0.0,8.0
5,3,1,52,NaN,5.0
6,3,2,41,-11.0,5.0
7,3,3,51,10.0,5.0
8,4,1,1373,NaN,10.0
9,4,2,1373,0.0,11.0
10,5,1,2371,NaN,7.0


### 4. Agregaciones por archivo y por mes

Para facilitar el análisis visual en el siguiente notebook, generaremos resúmenes estadísticos agrupando los datos:
1. **Resumen por archivo:** Analiza la historia completa de cada documento (fechas extremas, cambio neto de longitud y tiempos).
2. **Resumen mensual:** Agrupa las transiciones (versiones 2 en adelante) por archivo y por mes calendario para observar la magnitud de los cambios en el tiempo.

Finalmente, guardaremos estas tablas agregadas junto con los datos base en formato Parquet.

In [6]:
# 1. Agregación por Archivo
agregacion_archivo = df_analisis.groupby('source_markdown_file_history_id').agg(
    cantidad_versiones=('rank', 'max'),
    primera_fecha=('committed_at', 'min'),
    ultima_fecha=('committed_at', 'max'),
    longitud_inicial=('longitud_body', 'first'),
    longitud_final=('longitud_body', 'last'),
    mediana_tiempo_entre_versiones=('dias_desde_anterior', 'median')
).reset_index()

agregacion_archivo['cambio_neto_longitud'] = agregacion_archivo['longitud_final'] - agregacion_archivo['longitud_inicial']

# 2. Agregación por Archivo y Mes 
# Solo tomamos las transiciones (rank > 1, descartando la primera versión sin cambios)
transiciones = df_analisis[df_analisis['rank'] > 1].copy()

# Convertimos la fecha al primer día del mes para poder agrupar (ej. 2023-05-15 -> 2023-05-01)
transiciones['mes'] = transiciones['committed_at'].dt.to_period('M').dt.to_timestamp()

agregacion_mes = transiciones.groupby(['source_markdown_file_history_id', 'mes']).agg(
    cantidad_transiciones=('rank', 'count'),
    mediana_magnitud_cambio=('magnitud_cambio_body', 'median')
).reset_index()

print("¡Agregaciones calculadas con éxito!")

# 3. Guardar las 4 tablas solicitadas en formato Parquet
import os

# Aseguramos de que la ruta exista dependiendo de dónde abriste Jupyter
ruta_guardado = 'data/processed' 
if not os.path.exists(ruta_guardado):
    os.makedirs(ruta_guardado, exist_ok=True)

df_analisis.to_parquet(f'{ruta_guardado}/medidas_version.parquet', index=False)
transiciones.to_parquet(f'{ruta_guardado}/transiciones.parquet', index=False)
agregacion_archivo.to_parquet(f'{ruta_guardado}/agregacion_archivo.parquet', index=False)
agregacion_mes.to_parquet(f'{ruta_guardado}/agregacion_mes.parquet', index=False)

print(f"Archivos Parquet guardados exitosamente en la carpeta '{ruta_guardado}'")

/tmp/ipykernel_313269/1117775593.py:18: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  transiciones['mes'] = transiciones['committed_at'].dt.to_period('M').dt.to_timestamp()


¡Agregaciones calculadas con éxito!
Archivos Parquet guardados exitosamente en la carpeta 'data/processed'
